### **Reading & Loading Data**

In [ ]:
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import wordnet
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import random
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')


In [ ]:
data = pd.read_csv('/content/spam.csv' , encoding='latin-1')
data

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [ ]:
x = data['v2']
y = data['v1']

print(f"Loaded {len(x)} messages")
print(f"Class distribution:\n{y.value_counts()}")


Loaded 5572 messages
Class distribution:
v1
ham     4825
spam     747
Name: count, dtype: int64


In [ ]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### **STEP 2: PREPROCESSING**

In [ ]:
print("\n[2/6] Preprocessing text data...")

def preprocess_text(text):
    """Clean and preprocess text"""
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = ''.join([char for char in text if char not in string.punctuation])

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatize using WordNet
    from nltk.stem import WordNetLemmatizer
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return ' '.join(tokens)

# Apply preprocessing
data['clean_message'] = data['v2'].apply(preprocess_text)
print("Preprocessing complete")


[2/6] Preprocessing text data...
Preprocessing complete

[3/6] Splitting data...
Training set: 3900 messages
Validation set: 836 messages
Test set: 836 messages


### **STEP 3: TRAIN/VAL/TEST SPLIT**

In [ ]:
print("\n[3/6] Splitting data...")

X = data['clean_message'].values
y = (data['v1'] == 'spam').astype(int).values  # 1 for spam, 0 for ham

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Second split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Training set: {len(X_train)} messages")
print(f"Validation set: {len(X_val)} messages")
print(f"Test set: {len(X_test)} messages")

### **STEP 4: BASELINE MODEL (Logistic Regression)**

In [ ]:
print("\n[4/6] Training baseline model...")

# Vectorize text using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train Logistic Regression
baseline_model = LogisticRegression(random_state=42, C=1.0, max_iter=1000)
baseline_model.fit(X_train_vec, y_train)

# Evaluate baseline
y_pred_baseline = baseline_model.predict(X_test_vec)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

print(f"\nBaseline Model Results:")
print(f"Accuracy: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_baseline, target_names=['Ham', 'Spam']))



[4/6] Training baseline model...

Baseline Model Results:
Accuracy: 0.9689 (96.89%)

Classification Report:
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       724
        Spam       0.99      0.78      0.87       112

    accuracy                           0.97       836
   macro avg       0.98      0.89      0.93       836
weighted avg       0.97      0.97      0.97       836



### **STEP 5: MAD-LIB ATTACK IMPLEMENTATION**

In [ ]:
print("\n[5/6] Implementing Mad-Lib Attack...")

def get_synonyms(word):
    """Get synonyms for a word using WordNet"""
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            # Get lower case synonym, avoid underscores and same word
            synonym = lemma.name().lower()
            if synonym != word and '_' not in synonym:
                synonyms.add(synonym)
    return list(synonyms)

def mad_lib_attack(text, substitution_rate=0.3, preserve_meaning=True):
    """
    Apply Mad-Lib synonym substitution attack

    Args:
        text: Input text string
        substitution_rate: Probability of replacing each word (0.0 to 1.0)
        preserve_meaning: If True, use synonyms; if False, random words

    Returns:
        Attacked text string
    """
    words = text.split()
    if len(words) == 0:
        return text

    attacked_words = []

    for word in words:
        # Decide whether to replace this word
        if random.random() < substitution_rate:
            if preserve_meaning:
                # Get synonyms for the word
                synonyms = get_synonyms(word)
                if synonyms:
                    # Replace with random synonym
                    attacked_words.append(random.choice(synonyms))
                else:
                    attacked_words.append(word)
            else:
                # Just add some noise (typo simulation)
                if len(word) > 3:
                    # Random character substitution
                    pos = random.randint(0, len(word)-1)
                    word = word[:pos] + random.choice('abcdefghijklmnopqrstuvwxyz') + word[pos+1:]
                attacked_words.append(word)
        else:
            attacked_words.append(word)

    return ' '.join(attacked_words)

def apply_attack_to_dataset(messages, substitution_rate=0.3, attack_type='synonym'):
    """Apply Mad-Lib attack to entire dataset"""
    attacked_messages = []

    for msg in messages:
        if attack_type == 'synonym':
            attacked_msg = mad_lib_attack(msg, substitution_rate, preserve_meaning=True)
        elif attack_type == 'noise':
            attacked_msg = mad_lib_attack(msg, substitution_rate, preserve_meaning=False)
        else:
            attacked_msg = msg
        attacked_messages.append(attacked_msg)

    return attacked_messages

# Apply attack to test set
attack_type = 'synonym'  # Try 'synonym' or 'noise'
substitution_rate = 0.4  # Replace 40% of words

print(f"Attack configuration:")
print(f"  - Type: {attack_type}")
print(f"  - Substitution rate: {substitution_rate}")

# Generate attacked test set
X_test_attacked = apply_attack_to_dataset(X_test, substitution_rate, attack_type)

# Show examples of the attack
print("\nAttack Examples:")
print("-" * 50)
for i in range(min(3, len(X_test))):
    original = X_test[i]
    attacked = X_test_attacked[i]
    print(f"ORIGINAL: {original}")
    print(f"ATTACKED: {attacked}")
    print()

# Vectorize attacked test set using the SAME vectorizer
X_test_attacked_vec = vectorizer.transform(X_test_attacked)

# Evaluate model on attacked data
y_pred_attacked = baseline_model.predict(X_test_attacked_vec)
attacked_accuracy = accuracy_score(y_test, y_pred_attacked)

print(f"\nModel Performance After Attack:")
print(f"Clean accuracy: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print(f"Attacked accuracy: {attacked_accuracy:.4f} ({attacked_accuracy*100:.2f}%)")
print(f"Accuracy drop: {(baseline_accuracy - attacked_accuracy)*100:.2f}%")
#additional but more important


# Show specific examples where attack succeeded
print("\nAttack Success Examples (Spam misclassified as Ham):")
print("-" * 60)
spam_indices = np.where(y_test == 1)[0]
successful_attacks = 0
for idx in spam_indices[:10]:
    original_pred = baseline_model.predict(vectorizer.transform([X_test[idx]]))[0]
    attacked_pred = baseline_model.predict(vectorizer.transform([X_test_attacked[idx]]))[0]
    if original_pred == 1 and attacked_pred == 0:
        successful_attacks += 1
        print(f"Original: {X_test[idx][:50]}... -> Prediction: SPAM")
        print(f"Attacked: {X_test_attacked[idx][:50]}... -> Prediction: HAM (FAILED!)")
        print()


[5/6] Implementing Mad-Lib Attack...
Attack configuration:
  - Type: synonym
  - Substitution rate: 0.4

Attack Examples:
--------------------------------------------------
ORIGINAL: thank generally date brothas
ATTACKED: thank broadly engagement brothas

ORIGINAL: dear 0776xxxxxxx uve invited xchat final attempt contact u txt chat 86688 150pmsgrcvdhgsuite3422landsroww1j6hl ldn 18yrs
ATTACKED: dear 0776xxxxxxx uve invite xchat terminal attempt meet u txt chat 86688 150pmsgrcvdhgsuite3422landsroww1j6hl ldn 18yrs

ORIGINAL: call
ATTACKED: forebode


Model Performance After Attack:
Clean accuracy: 0.9689 (96.89%)
Attacked accuracy: 0.9282 (92.82%)
Accuracy drop: 4.07%

Attack Success Examples (Spam misclassified as Ham):
------------------------------------------------------------
Original: dear 0776xxxxxxx uve invited xchat final attempt c... -> Prediction: SPAM
Attacked: dear 0776xxxxxxx uve invite xchat terminal attempt... -> Prediction: HAM (FAILED!)

Original: sm service inclusive t

### **Defense1 against Mid lib**

In [ ]:
print("\n[6/6] Implementing Defense Against Mad-Lib Attack...")

# DEFENSE 1: Character-level features
print("\n--- DEFENSE 1: Character N-grams ---")

# Create character-level TF-IDF vectorizer
char_vectorizer = TfidfVectorizer(
    analyzer='char',           # Use character n-grams instead of words
    ngram_range=(3, 5),       # 3-5 character sequences
    max_features=5000
)

# Train on clean data with character features
X_train_char = char_vectorizer.fit_transform(X_train)
X_test_char = char_vectorizer.transform(X_test)
X_test_attacked_char = char_vectorizer.transform(X_test_attacked)

# Train defense model
defense_model_char = LogisticRegression(random_state=42, C=1.0, max_iter=1000)
defense_model_char.fit(X_train_char, y_train)

# Evaluate defense on attacked data
y_pred_defense = defense_model_char.predict(X_test_attacked_char)
defense_accuracy = accuracy_score(y_test, y_pred_defense)

print(f"Defense (Character N-grams) Performance:")
print(f"Clean accuracy: {accuracy_score(y_test, defense_model_char.predict(X_test_char)):.4f}")
print(f"Attacked accuracy: {defense_accuracy:.4f} ({defense_accuracy*100:.2f}%)")
print(f"Improvement over undefended: {(defense_accuracy - attacked_accuracy)*100:.2f}%")



[6/6] Implementing Defense Against Mad-Lib Attack...

--- DEFENSE 1: Character N-grams ---
Defense (Character N-grams) Performance:
Clean accuracy: 0.9737
Attacked accuracy: 0.9581 (95.81%)
Improvement over undefended: 2.99%


### **Defense 2 against Mid lib**

In [ ]:

# DEFENSE 2: Adversarial Training (Augment training with attacked examples)
print("\n--- DEFENSE 2: Adversarial Training ---")

# Generate attacked version of training data
X_train_attacked = apply_attack_to_dataset(X_train, substitution_rate, attack_type)

# Combine clean and attacked training data
X_train_augmented = np.concatenate([X_train, X_train_attacked])
y_train_augmented = np.concatenate([y_train, y_train])  # Labels stay the same

# Vectorize augmented training
X_train_augmented_vec = vectorizer.fit_transform(X_train_augmented)

# Train model on augmented data
defense_model_aug = LogisticRegression(random_state=42, C=1.0, max_iter=1000)
defense_model_aug.fit(X_train_augmented_vec, y_train_augmented)

# Evaluate on attacked test set
X_test_attacked_vec = vectorizer.transform(X_test_attacked)  # Re-transform
defense_aug_accuracy = accuracy_score(y_test, defense_model_aug.predict(X_test_attacked_vec))

print(f"Defense (Adversarial Training) Performance:")
print(f"Attacked accuracy: {defense_aug_accuracy:.4f} ({defense_aug_accuracy*100:.2f}%)")
print(f"Improvement over undefended: {(defense_aug_accuracy - attacked_accuracy)*100:.2f}%")



--- DEFENSE 2: Adversarial Training ---
Defense (Adversarial Training) Performance:
Attacked accuracy: 0.9581 (95.81%)
Improvement over undefended: 2.99%


### **Defense 3 against Midlib**

In [ ]:
# DEFENSE 3: Ensemble Defense (Best of both)
print("\n--- DENFENSE 3: Combined Defense (Character N-grams + Adversarial) ---")

# Generate attacked training for character model
X_train_char_attacked = apply_attack_to_dataset(X_train, substitution_rate, attack_type)

# Combine for character model
X_train_char_augmented = np.concatenate([X_train, X_train_char_attacked])
y_train_char_augmented = np.concatenate([y_train, y_train])

# Train character model on augmented data
X_train_char_augmented_vec = char_vectorizer.fit_transform(X_train_char_augmented)
defense_combined = LogisticRegression(random_state=42, C=1.0, max_iter=1000)
defense_combined.fit(X_train_char_augmented_vec, y_train_char_augmented)

# Evaluate
X_test_attacked_char = char_vectorizer.transform(X_test_attacked)
combined_accuracy = accuracy_score(y_test, defense_combined.predict(X_test_attacked_char))

print(f"Combined Defense Performance:")
print(f"Attacked accuracy: {combined_accuracy:.4f} ({combined_accuracy*100:.2f}%)")
print(f"Improvement over undefended: {(combined_accuracy - attacked_accuracy)*100:.2f}%")


--- DENFENSE 3: Combined Defense (Character N-grams + Adversarial) ---
Combined Defense Performance:
Attacked accuracy: 0.9725 (97.25%)
Improvement over undefended: 4.43%


### **Summary about each defense**

In [ ]:
print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

results = pd.DataFrame({
    'Model': ['Baseline (No Defense)', 'Under Attack', 'Defense 1: Character N-grams',
              'Defense 2: Adversarial Training', 'Defense 3: Combined'],
    'Accuracy': [baseline_accuracy, attacked_accuracy, defense_accuracy,
                 defense_aug_accuracy, combined_accuracy],
    'Improvement vs Attack': ['N/A', 'Baseline', f'{((defense_accuracy - attacked_accuracy)*100):.1f}%',
                              f'{((defense_aug_accuracy - attacked_accuracy)*100):.1f}%',
                              f'{((combined_accuracy - attacked_accuracy)*100):.1f}%']
})

print(results.to_string(index=False))

print("\n" + "=" * 60)
print("CONCLUSION")
print("=" * 60)
print(f"""
The Mad-Lib Attack successfully reduced model accuracy from
{baseline_accuracy*100:.1f}% to {attacked_accuracy*100:.1f}%.

Best Defense: {'Combined Defense' if combined_accuracy == max([defense_accuracy, defense_aug_accuracy, combined_accuracy])
               else 'Character N-grams' if defense_accuracy > defense_aug_accuracy
               else 'Adversarial Training'}

The defense works because:
1. Character n-grams capture sub-word patterns that synonyms can't hide
2. Adversarial training teaches the model to recognize attacked patterns
3. Combining both provides robust protection
""")


FINAL RESULTS SUMMARY
                          Model  Accuracy Improvement vs Attack
          Baseline (No Defense)  0.968900                   N/A
                   Under Attack  0.928230              Baseline
   Defense 1: Character N-grams  0.958134                  3.0%
Defense 2: Adversarial Training  0.958134                  3.0%
            Defense 3: Combined  0.972488                  4.4%

CONCLUSION

The Mad-Lib Attack successfully reduced model accuracy from 
96.9% to 92.8%.

Best Defense: Combined Defense

The defense works because:
1. Character n-grams capture sub-word patterns that synonyms can't hide
2. Adversarial training teaches the model to recognize attacked patterns
3. Combining both provides robust protection

